# 🌊 Python Stream Processing — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Think of a busy highway toll system. Cars (events) arrive at different rates. Each lane (partition) handles a subset of cars independently. Toll booth workers (consumers) read their lane, stamp a ticket (offset), and the car moves on. If a worker calls in sick, their lanes are redistributed to others. The highway never stops — cars that already passed can't come back, but you have the receipts (log) to replay.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Stream Processing? The Visual Model](#1) |
| 2 | [Core Concepts — Setup & Vocabulary](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Kafka Architecture — Topics, Partitions, Consumer Groups](#5) |
| 6 | [Pattern 2: Delivery Semantics — At-Most / At-Least / Exactly-Once](#6) |
| 7 | [Pattern 3: Windowing — Tumbling, Sliding, Session](#7) |
| 8 | [Pattern 4: Stream Processing Frameworks — Kafka Streams vs Flink vs Spark](#8) |
| 9 | [Pattern 5: Backpressure & Flow Control](#9) |
| 10 | [The Stream Processing Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. What Is Stream Processing? The Visual Model

---

```
KAFKA TOPIC: "user-clicks"  (3 partitions, replication-factor=2)

PRODUCERS                     BROKER CLUSTER
──────────                    ──────────────────────────────────────────────────
 App Server 1 ─────────────►  Partition 0: [msg0][msg1][msg4][msg7]  offset→3
 App Server 2 ─────────────►  Partition 1: [msg2][msg5][msg8]        offset→2
 App Server 3 ─────────────►  Partition 2: [msg3][msg6][msg9]        offset→2
                              ──────────────────────────────────────────────────
                              Leader + Followers (ISR replicas per partition)

CONSUMER GROUP: "analytics-group"
──────────────────────────────────────────────────────────────────────────────
 Consumer A  reads  Partition 0   (committed offset = 3 → next: msg at idx 3)
 Consumer B  reads  Partition 1   (committed offset = 2 → next: msg at idx 2)
 Consumer C  reads  Partition 2   (committed offset = 2 → next: msg at idx 2)
──────────────────────────────────────────────────────────────────────────────
 Rule: max 1 consumer per partition per group
 If Consumer A dies → rebalance: A's partition assigned to B or C

WINDOWING (on a stream of events with timestamps):

 Event stream: e1(t=1) e2(t=3) e3(t=5) e4(t=7) e5(t=9) e6(t=11)

 TUMBLING  (size=4):  [e1,e2,e3] | [e4,e5]    | [e6]      (no overlap)
 SLIDING   (size=4, step=2): [e1,e2,e3] [e2,e3,e4] [e3,e4,e5] ... (overlapping)
 SESSION   (gap=3):  [e1,e2,e3,e4,e5] | [e6]              (gap-based grouping)

DELIVERY SEMANTICS:

 At-most-once:   commit BEFORE processing  → possible loss, no duplicates
 At-least-once:  commit AFTER processing   → no loss, possible duplicates
 Exactly-once:   idempotent producer +
                 transactional API          → no loss, no duplicates

WHY PARTITIONS MATTER:
 Parallelism = number of partitions
 More partitions → more consumers → higher throughput
 Ordering guaranteed WITHIN a partition, NOT across partitions
 Partition key determines which partition a message lands in
```

<a id='2'></a>
## 2. Core Concepts — Setup & Vocabulary

In [ ]:
# Core stream processing vocabulary — all simulated in pure Python
# No external libraries needed; we model the mechanics directly

from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Callable, Any
import hashlib
import time

# ── Event: the atomic unit moving through a stream ──────────────────────────
@dataclass
class Event:
    key: str          # partition key — same key always lands in same partition
    value: Any        # the payload
    timestamp: float  # event time (not processing time)
    offset: int = 0   # position inside its partition (assigned by broker)

# ── Partition: ordered, immutable log ───────────────────────────────────────
@dataclass
class Partition:
    partition_id: int
    log: List[Event] = field(default_factory=list)  # append-only log
    hw_offset: int = 0  # high-watermark: highest committed offset across ISR

    def append(self, event: Event):
        event.offset = len(self.log)  # offset = position in log
        self.log.append(event)
        self.hw_offset = len(self.log)  # advance watermark after append

    def read_from(self, offset: int) -> List[Event]:
        return self.log[offset:]  # read from committed position onward

# ── Topic: collection of partitions ─────────────────────────────────────────
class Topic:
    def __init__(self, name: str, num_partitions: int):
        self.name = name
        self.partitions = [Partition(i) for i in range(num_partitions)]
        self.num_partitions = num_partitions

    def produce(self, key: str, value: Any, timestamp: float = None):
        # hash the key to pick a partition — same key always same partition
        ts = timestamp if timestamp else time.time()
        partition_id = int(hashlib.md5(key.encode()).hexdigest(), 16) % self.num_partitions
        event = Event(key=key, value=value, timestamp=ts)
        self.partitions[partition_id].append(event)
        return partition_id  # caller can see which partition was chosen

# ── ConsumerGroup: tracks offsets per partition ──────────────────────────────
class ConsumerGroup:
    def __init__(self, group_id: str, topic: Topic):
        self.group_id = group_id
        self.topic = topic
        # committed_offsets[p] = next offset to read from partition p
        self.committed_offsets = {p: 0 for p in range(topic.num_partitions)}

    def poll(self, partition_id: int) -> List[Event]:
        offset = self.committed_offsets[partition_id]
        return self.topic.partitions[partition_id].read_from(offset)

    def commit(self, partition_id: int, offset: int):
        # advance the committed cursor — messages before this offset won't replay
        self.committed_offsets[partition_id] = offset

# ── Smoke test ───────────────────────────────────────────────────────────────
topic = Topic("user-clicks", num_partitions=3)
for i in range(6):
    key = f"user_{i % 3}"  # 3 distinct keys → distribute across partitions
    pid = topic.produce(key, {"page": f"page_{i}"}, timestamp=float(i))
    print(f"  produced key={key} → partition {pid}")

print()
for p in topic.partitions:
    print(f"  Partition {p.partition_id}: {len(p.log)} events, hw={p.hw_offset}")

print("\nCore stream vocabulary defined.")

<a id='3'></a>
## 3. The Core API — All Operations

---

```
KAFKA / STREAM PROCESSING OPERATIONS
─────────────────────────────────────────────────────────────────────────────
OPERATION             COMPLEXITY   WHAT IT DOES
─────────────────────────────────────────────────────────────────────────────
produce(key, value)   O(1) amort   append event to partition (hash(key) % N)
poll(partition, off)  O(k)         fetch k events from given offset
commit(offset)        O(1)         advance consumer cursor (no replay below)
seek(offset)          O(1)         move cursor to arbitrary offset (replay)
rebalance()           O(P)         reassign partitions when group membership changes
window_assign(event)  O(1)         determine which window(s) the event belongs to
watermark_advance(t)  O(1)         signal that no event older than t will arrive
state_store.get(k)    O(1)         read local state (RocksDB / in-memory)
state_store.put(k,v)  O(1)         write local state (changelog topic for fault tol.)
─────────────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  Use wall-clock time as event time — out-of-order events will break windows
❌  Commit offset BEFORE processing — you lose events if consumer crashes mid-process
❌  Share state across partitions without a join/co-group — breaks parallelism
❌  Use a single partition for ordered global processing — kills throughput
❌  Rely on Kafka log for long-term storage — set retention and archive to S3/GCS
❌  Read from __consumer_offsets directly — use consumer group APIs
❌  Mix event time and processing time in the same window — pick one model
```

In [ ]:
# Live demo of core stream operations

topic = Topic("orders", num_partitions=3)
group = ConsumerGroup("order-processor", topic)

# PRODUCE: events land in partitions based on key hash
keys = ["customer_A", "customer_B", "customer_A", "customer_C", "customer_B"]
for i, k in enumerate(keys):
    pid = topic.produce(k, {"amount": (i+1) * 10}, timestamp=float(i))
    print(f"  produce key={k:12s} → partition {pid}")

print()
# POLL: each partition has its queue of events
for pid in range(3):
    events = group.poll(pid)
    print(f"  poll partition {pid}: {len(events)} events | {[e.key for e in events]}")

print()
# COMMIT: advance cursor — simulates at-least-once after processing
for pid in range(3):
    events = group.poll(pid)
    for e in events:
        pass  # simulate processing each event
    if events:
        group.commit(pid, len(events))  # commit past all processed events
        print(f"  committed partition {pid} to offset {len(events)}")

print()
# SEEK: replay from offset 0 — nothing comes back after commit unless we seek back
group.committed_offsets[0] = 0  # manually rewind partition 0 (simulate seek)
replayed = group.poll(0)
print(f"  after seek(0) on partition 0: {len(replayed)} events replayed")

print()
# REBALANCE SCENARIO: consumer dies, its partitions redistribute
print("  rebalance scenario:")
print("    before: Consumer A → p0, Consumer B → p1, Consumer C → p2")
print("    Consumer B crashes...")
print("    after:  Consumer A → p0 + p1, Consumer C → p2")
print("    (Consumer A will re-read p1 from last committed offset — no data loss)")

print("\nCore API demo complete.")

<a id='4'></a>
## 4. Decision Map — When To Use What

---

```
SIGNAL IN THE PROBLEM                           WHAT TO USE
────────────────────────────────────────────────────────────────────────────
"events arrive continuously, low latency"       streaming pipeline (Kafka + Flink)
"process in order per entity (user/session)"    partition by entity key
"count events in last N minutes"                tumbling or sliding window
"group by user session (idle timeout)"          session window
"late data may arrive after watermark"          allowed lateness + side output
"exactly-once semantics required"               idempotent producer + transactions
"at-least-once is fine, tolerate duplicates"    commit after processing + dedup key
"fast reads + historical replay"                Kafka (retention) + S3 archive
"join two streams"                              stream-stream join with state store
"enrich stream with static table"               stream-table join (KTable/broadcast)
"aggregate across partitions"                   use a single output partition or reduce
"consumer is too slow for producer"             backpressure — reduce poll batch size
"need SQL on streams"                           ksqlDB / Flink SQL / Spark Structured
"micro-batch is acceptable (seconds latency)"   Spark Structured Streaming
"true event-time processing with watermarks"    Apache Flink
"simple stateful ops within Kafka ecosystem"    Kafka Streams (no extra infra)
────────────────────────────────────────────────────────────────────────────
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Kafka Architecture — Topics, Partitions, Consumer Groups

---

```
PROBLEM:
  Design a system that can fan out messages from multiple producers to multiple
  consumer groups independently, with replay, ordering, and fault tolerance.

APPROACH:
  Kafka is a distributed commit log. Producers append to partition tails.
  Each consumer group independently tracks its own offset per partition.
  Rebalancing redistributes partitions when group membership changes.

SLOW MOTION TRACE (3-partition topic, 2 consumer groups):

  Topic "events" — 3 partitions
    P0: [e0, e3, e6]   (all events with hash(key)%3 == 0)
    P1: [e1, e4, e7]   (hash(key)%3 == 1)
    P2: [e2, e5, e8]   (hash(key)%3 == 2)

  Consumer Group A (3 consumers — 1:1 with partitions)
    A0 reads P0, offsets: 0→1→2→3
    A1 reads P1, offsets: 0→1→2→3
    A2 reads P2, offsets: 0→1→2→3

  Consumer Group B (1 consumer — reads ALL partitions)
    B0 reads P0+P1+P2, offsets tracked separately per partition

  A's offset changes do NOT affect B's offset — complete independence

  A0 crashes:
    Before: A0→P0, A1→P1, A2→P2
    After:  A1→P0+P1, A2→P2     (P0 replayed from A0's last committed offset)

KEY INSIGHT:
  Consumer groups are the fan-out mechanism — add a new group to get independent
  processing without touching producers or existing consumers.

TIME / SPACE:
  Produce:   O(1) amortized — append to partition tail
  Consume:   O(k) — read k messages from offset
  Rebalance: O(P) — P = number of partitions in the group
  Storage:   O(N * R) — N messages × replication factor
```

In [ ]:
# Pattern 1: Kafka Architecture simulation
# Demonstrates: partitioning, independent consumer groups, rebalance

# Slow motion on 9 events, 3 partitions, 2 consumer groups:
# step  event   key          partition   why
#  1    e0      user_0       P0          hash(user_0) % 3 == 0
#  2    e1      user_1       P1          hash(user_1) % 3 == 1
#  3    e2      user_2       P2          hash(user_2) % 3 == 2
#  4    e3      user_0       P0          same key → same partition (ordering!)
#  ...  each user always goes to the same partition

class KafkaArchitectureDemo:
    """
    Stream Processing Pattern 1 — Kafka Architecture
    Approach: Simulate topic partitioning and independent consumer group offsets.
    Time:  O(1) produce, O(k) consume
    Space: O(N) total events across all partitions
    """

    def __init__(self, topic_name: str, num_partitions: int):
        self.topic = Topic(topic_name, num_partitions)
        self.consumer_groups: Dict[str, ConsumerGroup] = {}

    def add_consumer_group(self, group_id: str) -> ConsumerGroup:
        # each group gets a fresh, independent offset tracker
        cg = ConsumerGroup(group_id, self.topic)
        self.consumer_groups[group_id] = cg
        return cg

    def produce_batch(self, events: List[tuple]) -> Dict[int, int]:
        # track how many events landed in each partition
        partition_counts = defaultdict(int)
        for key, value, ts in events:
            pid = self.topic.produce(key, value, timestamp=ts)
            partition_counts[pid] += 1
        return dict(partition_counts)

    def simulate_rebalance(self, group_id: str, dead_consumer_partitions: List[int]):
        # simulates a consumer dying — other consumers pick up its partitions
        cg = self.consumer_groups[group_id]
        print(f"  [rebalance] group={group_id}, reassigning partitions {dead_consumer_partitions}")
        for pid in dead_consumer_partitions:
            committed = cg.committed_offsets[pid]
            available = cg.topic.partitions[pid].hw_offset
            backlog = available - committed
            print(f"    partition {pid}: committed={committed}, hw={available}, backlog={backlog} msgs")


demo = KafkaArchitectureDemo("page-views", num_partitions=3)

# produce 9 events — 3 users, each appears 3 times
events_to_produce = [
    (f"user_{u}", {"page": f"p{t}"}, float(t * 3 + u))
    for t in range(3) for u in range(3)
]
counts = demo.produce_batch(events_to_produce)
print("Events per partition after producing 9 events:")
for pid, cnt in sorted(counts.items()):
    keys_in_partition = [e.key for e in demo.topic.partitions[pid].log]
    print(f"  P{pid}: {cnt} events — keys: {list(dict.fromkeys(keys_in_partition))}")

# add two independent consumer groups
analytics = demo.add_consumer_group("analytics")
billing   = demo.add_consumer_group("billing")

# analytics processes and commits all partitions
print("\nanalytics group — processing all partitions:")
for pid in range(3):
    events = analytics.poll(pid)
    analytics.commit(pid, len(events))
    print(f"  analytics processed+committed {len(events)} events from P{pid}")

# billing has NOT polled yet — its offsets are still at 0
print("\nbilling group — checking pending events:")
for pid in range(3):
    pending = billing.poll(pid)
    print(f"  billing P{pid} pending: {len(pending)} (committed={billing.committed_offsets[pid]})")

# simulate rebalance: analytics consumer for P0 dies
print("\nSimulating consumer crash in analytics group:")
demo.simulate_rebalance("analytics", dead_consumer_partitions=[0])

print("\nKafka architecture demo complete.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Delivery Semantics — At-Most / At-Least / Exactly-Once

---

```
PROBLEM:
  A consumer processes payments from a Kafka topic. What happens if it crashes
  mid-batch? Define the three delivery guarantees and their tradeoffs.

APPROACH:
  The position of the offset commit relative to processing determines semantics.
  Exactly-once requires idempotent producer + transactional commit on consumer side.

SLOW MOTION TRACE (batch of 3 events, crash after event 2):

  AT-MOST-ONCE:
    commit(offset=3) BEFORE processing e0, e1, e2
    process e0 ✓, process e1 ✓, CRASH before e2
    on restart: poll from offset=3 → e2 is LOST forever

  AT-LEAST-ONCE:
    process e0 ✓, process e1 ✓, CRASH before commit
    on restart: poll from offset=0 → e0, e1 replayed
    result: e0 and e1 processed TWICE (duplicates)

  EXACTLY-ONCE:
    producer: idempotent (sequence numbers, dedup on broker)
    consumer: atomic commit offset + output in one transaction
    CRASH at any point → rollback transaction → no duplicate, no loss
    cost: ~20% throughput overhead for transaction coordination

KEY INSIGHT:
  At-least-once + idempotent downstream write = effective exactly-once
  (cheaper than true Kafka transactions — prefer this when downstream supports it)

TIME / SPACE:
  At-most-once:   O(1) commit overhead — fastest, accept loss
  At-least-once:  O(1) commit overhead — add dedup key for idempotency
  Exactly-once:   O(log N) per msg — transaction coordinator overhead
```

In [ ]:
# Pattern 2: Delivery semantics — simulate all three guarantees

# Slow motion: crash happens after processing event index 1 (before commit)
# at-most-once:  committed before crash → e2 lost on restart
# at-least-once: not committed → e0,e1 replayed on restart
# exactly-once:  transaction rolled back → clean restart from last committed

class DeliverySemanticsSim:
    """
    Stream Processing Pattern 2 — Delivery Semantics
    Approach: Simulate at-most-once, at-least-once, exactly-once with crash injection.
    Time:  O(k) per batch of k events
    Space: O(k) for in-flight batch
    """

    def __init__(self, topic: Topic, partition_id: int = 0):
        self.topic = topic
        self.partition_id = partition_id
        self.committed_offset = 0
        self.processed_log = []  # track what was actually processed
        self.output_store: Dict[str, Any] = {}  # simulate downstream sink

    def _poll(self) -> List[Event]:
        return self.topic.partitions[self.partition_id].read_from(self.committed_offset)

    def at_most_once(self, crash_after: int = -1):
        # commit FIRST then process — fast, but drops messages on crash
        events = self._poll()
        batch_size = len(events)
        self.committed_offset += batch_size  # commit entire batch upfront
        for i, e in enumerate(events):
            if i == crash_after:
                print(f"    CRASH after {i} events — offset already committed to {self.committed_offset}")
                return  # crash: remaining events in batch are gone
            self.processed_log.append(e.value)
        print(f"    processed {len(self.processed_log)} events, offset={self.committed_offset}")

    def at_least_once(self, crash_after: int = -1):
        # process FIRST then commit — safe but may duplicate on restart
        events = self._poll()
        for i, e in enumerate(events):
            if i == crash_after:
                print(f"    CRASH after {i} events — offset NOT committed (still at {self.committed_offset})")
                return  # crash: offset not advanced, events will replay
            self.processed_log.append(e.value)
        self.committed_offset += len(events)  # commit only after all processed
        print(f"    processed {len(self.processed_log)} events, offset={self.committed_offset}")

    def exactly_once(self, crash_after: int = -1):
        # transactional: write output + advance offset atomically
        # if crash mid-transaction, rollback — no partial state visible
        events = self._poll()
        staged_output = []  # staging area — not committed to sink yet
        staged_offset = self.committed_offset
        for i, e in enumerate(events):
            if i == crash_after:
                print(f"    CRASH mid-transaction — rolling back staged offset {staged_offset + i}")
                return  # transaction rolled back: staged_output discarded
            staged_output.append(e.value)
            staged_offset += 1
        # atomic commit: both sink write and offset advance succeed together
        self.processed_log.extend(staged_output)
        self.committed_offset = staged_offset
        print(f"    transaction committed: {len(staged_output)} events, offset={self.committed_offset}")


# Build a topic with 5 events
t = Topic("payments", num_partitions=1)
for i in range(5):
    t.produce("customer_X", {"payment_id": i, "amount": (i+1)*100}, timestamp=float(i))

print("=== AT-MOST-ONCE (crash after event 2) ===")
sim1 = DeliverySemanticsSim(t)
sim1.at_most_once(crash_after=2)
print(f"  on restart: next poll starts at offset={sim1.committed_offset} — events 0-1 lost!")

print("\n=== AT-LEAST-ONCE (crash after event 2) ===")
sim2 = DeliverySemanticsSim(t)
sim2.at_least_once(crash_after=2)
print(f"  on restart: re-poll from offset={sim2.committed_offset} — events 0-1 will replay (duplicates!)")
# simulate restart and re-process
sim2.at_least_once(crash_after=-1)  # no crash this time
print(f"  total processed (including duplicates): {len(sim2.processed_log)} events")

print("\n=== EXACTLY-ONCE (crash after event 2) ===")
sim3 = DeliverySemanticsSim(t)
sim3.exactly_once(crash_after=2)
print(f"  on restart: clean state, offset={sim3.committed_offset}, no duplicates, no loss")
sim3.exactly_once(crash_after=-1)
print(f"  total processed (exact): {len(sim3.processed_log)} events")

print("\nDelivery semantics demo complete.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Windowing — Tumbling, Sliding, Session

---

```
PROBLEM:
  You have a stream of events with timestamps. You need to compute aggregates
  (count, sum, max) over groups of events based on time.

APPROACH:
  Three window types cover all use cases:
  - Tumbling: non-overlapping fixed buckets ("per minute")
  - Sliding: overlapping buckets ("last N minutes, re-evaluated every M minutes")
  - Session: gap-triggered grouping ("user session ends after 30 min idle")

SLOW MOTION TRACE (events at t=1,3,5,7,9,11,12):

  TUMBLING window (size=4):
    window [0,4):  events at t=1,3    → count=2
    window [4,8):  events at t=5,7    → count=2
    window [8,12): events at t=9,11   → count=2
    window [12,16):events at t=12     → count=1
    no event appears in two windows

  SLIDING window (size=4, step=2):
    window [0,4):  t=1,3              → count=2
    window [2,6):  t=3,5              → count=2  (events 3,5 shared with adjacent)
    window [4,8):  t=5,7              → count=2
    ... each event appears in ceil(size/step) windows

  SESSION window (gap=3):
    t=1: new session start
    t=3: gap=2 < 3 → extend session to [1,6) (3+3)
    t=5: gap=2 < 3 → extend session to [5,8) → merged: [1,8)
    t=7: gap=2 < 3 → extend to [7,10)
    t=9: gap=2 < 3 → extend
    t=11: gap=2 < 3 → extend
    t=12: gap=1 < 3 → extend
    → all in one session! (dense activity)

KEY INSIGHT:
  Watermarks tell the engine "no event older than T will arrive" — this is when
  the engine can safely close and emit a window's result.

TIME / SPACE:
  Tumbling: O(1) assign — each event in exactly one window
  Sliding:  O(size/step) — each event in multiple windows
  Session:  O(1) amort — merge sessions on arrival
  Space:    O(W * k) — W active windows × k events per window
```

In [ ]:
# Pattern 3: Windowing — Tumbling, Sliding, Session

# Slow motion on timestamps [1, 3, 5, 7, 9, 11, 12, 15, 20]:
# tumbling(size=4): [0,4)→{1,3}  [4,8)→{5,7}  [8,12)→{9,11}  [12,16)→{12,15}  [16,20)→ ...
# session(gap=4):   t=1,3,5,7,9,11,12,15 all within gap=4 of each other → 1 session
#                   t=20 is 5 away from t=15 → new session

class WindowingDemo:
    """
    Stream Processing Pattern 3 — Windowing
    Approach: Assign events to tumbling, sliding, or session windows by timestamp.
    Time:  O(N) for tumbling/session, O(N * size/step) for sliding
    Space: O(W * k) where W = active windows, k = events per window
    """

    @staticmethod
    def tumbling(events: List[float], window_size: float) -> Dict[str, List[float]]:
        # each event belongs to exactly one bucket: floor(t / size) * size
        buckets: Dict[float, List[float]] = defaultdict(list)
        for t in events:
            bucket_start = (t // window_size) * window_size
            buckets[bucket_start].append(t)
        # return as labeled windows
        result = {}
        for start in sorted(buckets):
            end = start + window_size
            label = f"[{start:.0f},{end:.0f})"
            result[label] = buckets[start]
        return result

    @staticmethod
    def sliding(events: List[float], window_size: float, step: float) -> Dict[str, List[float]]:
        # each event belongs to multiple buckets — one per step that covers it
        # find all window starts that include each event: from (t - size + step) to t in steps
        all_events = sorted(events)
        if not all_events:
            return {}
        max_t = all_events[-1]
        buckets: Dict[float, List[float]] = defaultdict(list)
        # generate all possible window starts from 0 to max_t
        start = 0.0
        while start <= max_t:
            end = start + window_size
            for t in all_events:
                if start <= t < end:
                    buckets[start].append(t)
            start += step
        result = {}
        for s in sorted(buckets):
            if buckets[s]:  # skip empty windows
                label = f"[{s:.0f},{s+window_size:.0f})"
                result[label] = buckets[s]
        return result

    @staticmethod
    def session(events: List[float], gap_timeout: float) -> List[List[float]]:
        # sort events; start new session whenever gap > timeout
        sorted_events = sorted(events)
        if not sorted_events:
            return []
        sessions = []
        current_session = [sorted_events[0]]
        for t in sorted_events[1:]:
            if t - current_session[-1] > gap_timeout:
                sessions.append(current_session)  # close session on gap
                current_session = [t]              # open new session
            else:
                current_session.append(t)  # extend current session
        sessions.append(current_session)  # close final session
        return sessions


timestamps = [1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 12.0, 15.0, 20.0]
wdemo = WindowingDemo()

print("=== TUMBLING WINDOW (size=4) ===")
tumbling_result = wdemo.tumbling(timestamps, window_size=4)
for window, evts in tumbling_result.items():
    print(f"  {window}: count={len(evts)} events={evts}")

print("\n=== SLIDING WINDOW (size=4, step=2) ===")
sliding_result = wdemo.sliding(timestamps, window_size=4, step=2)
for window, evts in sliding_result.items():
    print(f"  {window}: count={len(evts)} events={evts}")

print("\n=== SESSION WINDOW (gap_timeout=4) ===")
session_result = wdemo.session(timestamps, gap_timeout=4)
for i, session in enumerate(session_result):
    print(f"  Session {i+1}: count={len(session)} span=[{session[0]},{session[-1]}] events={session}")

print("\nWindowing demo complete.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Stream Processing Frameworks — Kafka Streams vs Flink vs Spark Structured Streaming

---

```
PROBLEM:
  You need to process a high-volume event stream with stateful aggregations,
  windowing, and joins. Which framework do you pick and why?

APPROACH:
  Three dominant frameworks, each with a distinct processing model:

  ┌──────────────────────────────────────────────────────────────────────┐
  │  KAFKA STREAMS          │  APACHE FLINK       │  SPARK STRUCT. STREAM │
  ├──────────────────────────────────────────────────────────────────────┤
  │  Java library (no cluster│  Distributed cluster│  Micro-batch (100ms+) │
  │  needed — runs in app)  │  true event-at-a-time│  on Spark cluster     │
  │  Kafka in + Kafka out   │  any source/sink    │  any source/sink      │
  │  KTable for local state │  RocksDB state store│  Spark SQL on streams │
  │  DSL + PAPI             │  DataStream / Table │  DataFrame API        │
  │  EOS via transactions   │  EOS checkpoint     │  EOS via idempotent   │
  │  ms latency             │  ms latency         │  100ms–1s latency     │
  └──────────────────────────────────────────────────────────────────────┘

SLOW MOTION: picking the right framework

  Signal: "already on Kafka, simple stateful ops, no cluster budget"
  → Kafka Streams: embedded library, auto-scales via partitions

  Signal: "complex event time, watermarks, late data, stream-stream join"
  → Flink: richest event-time support, SavePoints for upgrades

  Signal: "team knows Spark SQL, need historical backfill + streaming"
  → Spark Structured Streaming: unified batch+stream API

  Signal: "need SQL on streams right now, no code"
  → ksqlDB (Kafka) or Flink SQL

KEY INSIGHT:
  Framework choice is driven by: cluster budget, latency SLA, team skill,
  and whether you need true event-time semantics vs micro-batch.

TIME / SPACE:
  Kafka Streams: O(1) amort per event — local RocksDB for state
  Flink:         O(1) per event — distributed state with async checkpoints
  Spark SS:      O(k) per micro-batch — batch overhead per trigger interval
```

In [ ]:
# Pattern 4: Framework comparison simulation
# Models the key behavioral differences without actual Flink/Spark/Kafka installs

# Slow motion: event arrives at t=5, watermark is at t=3
# Flink:   process immediately (event-at-a-time), update window state
# Spark SS: buffer event until trigger fires at t+100ms, then process batch
# Kafka Streams: process in same thread, update local KTable

from typing import Tuple

@dataclass
class StreamEvent:
    key: str
    value: float
    event_time: float
    processing_time: float  # when framework receives it


class KafkaStreamsProcessor:
    """
    Kafka Streams model: embedded, local state, event-at-a-time, Kafka in/out.
    Approach: Each event updates a local KTable aggregation immediately.
    Time:  O(1) per event — local state access
    Space: O(K) where K = distinct keys in state store
    """
    def __init__(self):
        self.ktable: Dict[str, float] = {}  # local state store (RocksDB in prod)
        self.processed_count = 0

    def process(self, event: StreamEvent):
        # update aggregation immediately — no buffering
        self.ktable[event.key] = self.ktable.get(event.key, 0.0) + event.value
        self.processed_count += 1

    def get_state(self) -> Dict[str, float]:
        return dict(self.ktable)


class FlinkProcessor:
    """
    Flink model: true event-time, watermarks gate window closes, state backend.
    Approach: Windows accumulate; emit when watermark advances past window end.
    Time:  O(1) per event, O(W) per window close (emit aggregation)
    Space: O(W * k) where W = open windows, k = events per window
    """
    def __init__(self, window_size: float, allowed_lateness: float = 0.0):
        self.window_size = window_size
        self.allowed_lateness = allowed_lateness
        self.open_windows: Dict[float, List[StreamEvent]] = defaultdict(list)
        self.watermark = float('-inf')
        self.emitted: List[Tuple[str, float, List[StreamEvent]]] = []

    def _window_start(self, t: float) -> float:
        return (t // self.window_size) * self.window_size

    def process(self, event: StreamEvent):
        ws = self._window_start(event.event_time)
        # late event check: discard if past allowed_lateness
        if event.event_time < self.watermark - self.allowed_lateness:
            return  # too late — dropped (or routed to side output in real Flink)
        self.open_windows[ws].append(event)

    def advance_watermark(self, new_watermark: float):
        # watermark advance triggers window closes for expired windows
        self.watermark = new_watermark
        windows_to_close = [
            ws for ws in list(self.open_windows.keys())
            if ws + self.window_size <= new_watermark - self.allowed_lateness
        ]
        for ws in windows_to_close:
            events = self.open_windows.pop(ws)
            total = sum(e.value for e in events)
            label = f"[{ws:.0f},{ws+self.window_size:.0f})"
            self.emitted.append((label, total, events))


class SparkStructuredStreamingProcessor:
    """
    Spark Structured Streaming: micro-batch, trigger interval, DataFrame API.
    Approach: Buffer events until trigger fires, then process entire batch at once.
    Time:  O(k) per trigger — processes k buffered events
    Space: O(k) buffer — bounded by trigger interval × event rate
    """
    def __init__(self, trigger_interval: float):
        self.trigger_interval = trigger_interval
        self.buffer: List[StreamEvent] = []
        self.last_trigger = 0.0
        self.micro_batch_results: List[Dict] = []

    def process(self, event: StreamEvent):
        self.buffer.append(event)  # events accumulate until trigger
        # trigger fires when processing time passes interval boundary
        if event.processing_time >= self.last_trigger + self.trigger_interval:
            self._run_micro_batch(event.processing_time)

    def _run_micro_batch(self, processing_time: float):
        # process entire buffer as a mini-DataFrame operation
        if not self.buffer:
            return
        agg: Dict[str, float] = defaultdict(float)
        for e in self.buffer:
            agg[e.key] += e.value
        self.micro_batch_results.append({
            "trigger_time": processing_time,
            "batch_size": len(self.buffer),
            "aggregations": dict(agg)
        })
        self.buffer.clear()
        self.last_trigger = processing_time


# Generate shared event stream
events = [
    StreamEvent("user_A", 10.0, event_time=1.0, processing_time=1.1),
    StreamEvent("user_B", 20.0, event_time=2.0, processing_time=2.1),
    StreamEvent("user_A", 15.0, event_time=3.0, processing_time=3.2),
    StreamEvent("user_C", 30.0, event_time=4.0, processing_time=4.0),
    StreamEvent("user_B", 25.0, event_time=5.0, processing_time=5.1),
    StreamEvent("user_A",  5.0, event_time=6.0, processing_time=6.0),
    StreamEvent("user_C", 10.0, event_time=7.0, processing_time=7.2),
    StreamEvent("user_B", 35.0, event_time=8.0, processing_time=8.1),
]

# ── Kafka Streams ────────────────────────────────────────────────────────────
ks = KafkaStreamsProcessor()
for e in events:
    ks.process(e)
print("=== KAFKA STREAMS (running aggregate) ===")
print(f"  KTable state: {ks.get_state()}")
print(f"  processed {ks.processed_count} events individually (no buffering)")

# ── Flink ────────────────────────────────────────────────────────────────────
flink = FlinkProcessor(window_size=4.0, allowed_lateness=0.5)
for e in events:
    flink.process(e)
flink.advance_watermark(4.5)  # close window [0,4)
flink.advance_watermark(9.0)  # close window [4,8)
print("\n=== FLINK (event-time windows, watermark-triggered) ===")
for label, total, evts in flink.emitted:
    print(f"  window {label}: sum={total:.1f}, events={[e.key for e in evts]}")

# ── Spark Structured Streaming ───────────────────────────────────────────────
spark_ss = SparkStructuredStreamingProcessor(trigger_interval=3.0)
for e in events:
    spark_ss.process(e)
if spark_ss.buffer:  # force-flush remaining
    spark_ss._run_micro_batch(events[-1].processing_time + spark_ss.trigger_interval)
print("\n=== SPARK STRUCTURED STREAMING (micro-batch every 3s) ===")
for batch in spark_ss.micro_batch_results:
    print(f"  trigger@t={batch['trigger_time']:.1f}: batch_size={batch['batch_size']}, agg={batch['aggregations']}")

print("\nFramework comparison demo complete.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Backpressure & Flow Control

---

```
PROBLEM:
  A producer emits 10,000 events/sec. The consumer can only process 6,000/sec.
  Design a system that handles this without dropping events or crashing.

APPROACH:
  Backpressure = the consumer telling the producer to slow down.
  Three strategies:
    1. Buffering:   accept events into a bounded queue; slow down when full
    2. Dropping:    shed load — drop or sample events when overwhelmed
    3. Rate Control: producer-side throttle — slow production to match consumer

  In Kafka: consumer controls its own rate via max.poll.records and poll interval
  In Flink: async checkpoint mechanism pauses input when state is under pressure
  In Reactive Streams (RxJava/Akka): demand signals flow upstream from subscriber

SLOW MOTION TRACE (bounded buffer, capacity=5):

  t=0: producer pushes 3 → buffer=[1,2,3]  (3/5 full)
  t=1: consumer pops 1  → buffer=[2,3]     (2/5 full)
  t=2: producer pushes 4 → buffer=[2,3,4,5,6]  (5/5 FULL)
  t=3: producer tries push 7 → BLOCKED (backpressure signal sent upstream)
  t=3: consumer pops 1  → buffer=[3,4,5,6]    (4/5 → unblock producer)
  t=4: producer resumes → push 7 → buffer=[3,4,5,6,7]

  With DROP strategy:
  t=3: buffer full → event 7 is DROPPED (counts toward error SLA)
  t=3: send to DLQ (dead letter queue) for later inspection

KEY INSIGHT:
  Backpressure converts an unbounded overflow into a latency tradeoff.
  Always instrument: buffer fill %, consumer lag, drop rate — these are your SLAs.

TIME / SPACE:
  Buffer check:   O(1) — compare size vs capacity
  Rate control:   O(1) — token bucket or leaky bucket per interval
  Space:          O(B) — B = buffer capacity (must be bounded)
```

In [ ]:
# Pattern 5: Backpressure & Flow Control simulation

# Slow motion: producer emits 10 events, consumer processes 6 per round
# buffer capacity = 5: demonstrate block, drop, and rate-limit strategies

from enum import Enum

class BackpressureStrategy(Enum):
    BLOCK  = "block"   # wait until space is available
    DROP   = "drop"    # discard the event, count as lost
    DLQ    = "dlq"     # route to dead letter queue instead of dropping silently


class BoundedBuffer:
    """
    Stream Processing Pattern 5 — Backpressure
    Approach: Bounded queue with configurable overflow strategy; tracks metrics.
    Time:  O(1) push/pop
    Space: O(capacity) — bounded buffer prevents unbounded memory growth
    """

    def __init__(self, capacity: int, strategy: BackpressureStrategy):
        self.capacity = capacity
        self.strategy = strategy
        self.buffer: deque = deque()
        self.dropped = 0
        self.dlq: List[Any] = []
        self.blocked_events = 0
        self.total_produced = 0
        self.total_consumed = 0

    def push(self, event: Any) -> bool:
        # returns True if event was accepted, False if dropped
        self.total_produced += 1
        if len(self.buffer) < self.capacity:
            self.buffer.append(event)  # room available — accept immediately
            return True
        # buffer full — apply backpressure strategy
        if self.strategy == BackpressureStrategy.BLOCK:
            self.blocked_events += 1
            # in a real system this would block the producer thread
            # here we simulate by NOT adding — producer must retry later
            return False
        elif self.strategy == BackpressureStrategy.DROP:
            self.dropped += 1  # silent drop — count against loss SLA
            return False
        elif self.strategy == BackpressureStrategy.DLQ:
            self.dlq.append(event)  # route to dead letter queue
            self.dropped += 1       # counts as not processed in main pipeline
            return False
        return False

    def consume_batch(self, n: int) -> List[Any]:
        # consumer pulls up to n events per polling cycle
        batch = []
        for _ in range(min(n, len(self.buffer))):
            batch.append(self.buffer.popleft())
        self.total_consumed += len(batch)
        return batch

    def fill_pct(self) -> float:
        return 100.0 * len(self.buffer) / self.capacity

    def metrics(self) -> str:
        lag = self.total_produced - self.total_consumed - self.dropped
        return (
            f"produced={self.total_produced} consumed={self.total_consumed} "
            f"dropped={self.dropped} dlq={len(self.dlq)} "
            f"buffer={len(self.buffer)}/{self.capacity} lag={lag}"
        )


def simulate_pressure(strategy: BackpressureStrategy, label: str):
    buf = BoundedBuffer(capacity=5, strategy=strategy)
    print(f"\n=== {label} (capacity=5, produce=10/round, consume=6/round) ===")
    for rnd in range(3):
        # producer pushes 10 events
        accepted = 0
        for i in range(10):
            ok = buf.push(f"event_{rnd}_{i}")
            if ok:
                accepted += 1
        # consumer processes 6
        consumed = buf.consume_batch(6)
        print(f"  round {rnd+1}: produced=10 accepted={accepted} consumed={len(consumed)} "
              f"buffer_fill={buf.fill_pct():.0f}%")
    print(f"  final: {buf.metrics()}")


simulate_pressure(BackpressureStrategy.DROP,  "STRATEGY: DROP")
simulate_pressure(BackpressureStrategy.DLQ,   "STRATEGY: DLQ")
simulate_pressure(BackpressureStrategy.BLOCK, "STRATEGY: BLOCK")

print("\n--- Token Bucket Rate Limiter ---")
# token bucket: producer earns tokens at a fixed rate
# each event costs 1 token; no token = blocked
class TokenBucket:
    def __init__(self, rate: float, capacity: float):
        self.tokens = capacity      # start full
        self.capacity = capacity
        self.rate = rate            # tokens added per time unit
        self.last_refill = 0.0

    def consume(self, current_time: float, amount: float = 1.0) -> bool:
        elapsed = current_time - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_refill = current_time
        if self.tokens >= amount:
            self.tokens -= amount  # spend the token
            return True
        return False  # no token — rate limit applied

bucket = TokenBucket(rate=6.0, capacity=10.0)  # 6 events/sec, burst=10
allowed = 0
for t in range(20):  # simulate 20 events arriving at t=0,1,...19
    if bucket.consume(float(t)):
        allowed += 1
print(f"  token bucket: {allowed}/20 events allowed through at rate=6/sec")

print("\nBackpressure demo complete.")

<a id='10'></a>
## 10. The Stream Processing Decision Map

---

```
QUESTION TYPE                              KEY TECHNIQUE              PATTERN
────────────────────────────────────────────────────────────────────────────────
Fan-out to multiple teams independently    Consumer Groups            Pattern 1
Order guarantee per entity (user/order)    Partition by entity key    Pattern 1
Replay historical data                     seek(offset=0)             Pattern 1
Scale consumer throughput                  Add partitions             Pattern 1
Consumer crashes, no data loss             At-least-once + commit     Pattern 2
Payment dedup, no double-charge            Exactly-once / idempotent  Pattern 2
Fast fire-and-forget metrics               At-most-once               Pattern 2
Count events per minute (non-overlapping)  Tumbling window            Pattern 3
Rolling 5-min average, updated every min   Sliding window             Pattern 3
User session activity grouping             Session window             Pattern 3
Handle late arriving events                Watermark + allowed_late   Pattern 3
Simple stateful ops in Kafka ecosystem     Kafka Streams / KTable     Pattern 4
Complex event-time, joins, late data       Apache Flink               Pattern 4
Unified batch + streaming, Spark SQL       Spark Structured Streaming Pattern 4
Consumer lag behind producer               Backpressure + buffer      Pattern 5
Overflow events for manual review          Dead Letter Queue (DLQ)    Pattern 5
Burst traffic, sustained rate limit        Token Bucket / Rate Limit  Pattern 5
Monitor pipeline health                    Consumer lag metric        Pattern 5
────────────────────────────────────────────────────────────────────────────────

FRAMEWORK SELECTION MATRIX:

                    KAFKA STREAMS    FLINK           SPARK SS
Cluster needed?     No               Yes             Yes
Event-time support  Basic            Rich (best)     Good
State management    KTable/RocksDB   RocksDB + ckpt  Spark state
SQL support         ksqlDB           Flink SQL       Spark SQL
Latency             ms               ms              100ms–1s
EOS                 Yes (transact.)  Yes (checkpoint) Yes (idempot.)
Best for            Kafka-only apps  Complex streams  Spark teams

DELIVERY SEMANTICS QUICK PICK:

  Can tolerate loss?           → AT-MOST-ONCE  (metrics, non-critical events)
  Downstream is idempotent?    → AT-LEAST-ONCE (safe duplicates: DB upsert, dedup key)
  Financial / audit / legal?   → EXACTLY-ONCE  (pay the latency cost)
```

<a id='11'></a>
## 11. Interview Cheat Sheet

---

### When to reach for Stream Processing:

| Signal | What to Do |
|--------|------------|
| "Real-time dashboard" | Kafka → Flink/Kafka Streams → serving layer |
| "Process events as they arrive" | Streaming pipeline (not batch ETL) |
| "Order matters per user" | Partition by user_id key |
| "User sessions / activity windows" | Session windows with gap timeout |
| "Aggregate last N minutes" | Sliding window |
| "Hourly/daily rollup" | Tumbling window |
| "Consumer can't keep up" | Backpressure + DLQ or scale partitions |
| "No duplicates" | Exactly-once or idempotent downstream |
| "Late events must be handled" | Watermarks + allowed lateness |
| "Join two event streams" | Stream-stream join (Flink / Kafka Streams) |

---

### The O(1) operations — memorize these:

```python
# Partition assignment — same key always same partition
partition_id = hash(key) % num_partitions

# Tumbling window bucket
window_start = (event_time // window_size) * window_size

# Consumer offset commit (at-least-once)
group.commit(partition_id, current_offset + 1)

# Session window: close when gap > timeout
if current_event_time - last_event_time > gap_timeout:
    close_session(); open_new_session()

# Token bucket
tokens = min(capacity, tokens + elapsed * rate)
allowed = tokens >= cost
if allowed: tokens -= cost
```

---

### Common templates:

```python
# PATTERN: AT-LEAST-ONCE CONSUMER LOOP
while True:
    events = consumer.poll(partition_id)
    for event in events:
        process(event)               # process first
    consumer.commit(last_offset + 1) # commit after all done

# PATTERN: TUMBLING WINDOW AGGREGATION
def tumble(events, window_size):
    buckets = defaultdict(list)
    for e in events:
        start = (e.timestamp // window_size) * window_size
        buckets[start].append(e)
    return buckets

# PATTERN: SESSION WINDOW
def sessionize(events, gap):
    events = sorted(events, key=lambda e: e.timestamp)
    sessions, cur = [], [events[0]]
    for e in events[1:]:
        if e.timestamp - cur[-1].timestamp > gap:
            sessions.append(cur); cur = []
        cur.append(e)
    sessions.append(cur)
    return sessions

# PATTERN: BACKPRESSURE WITH BOUNDED QUEUE
buffer = deque(maxlen=CAPACITY)  # maxlen auto-drops oldest on overflow
buffer.append(event)             # never blocks — maxlen enforces bound

# PATTERN: CONSUMER GROUP OFFSET TRACKING
offsets = {partition_id: 0 for partition_id in range(num_partitions)}
events  = topic.partitions[pid].log[offsets[pid]:]
offsets[pid] += len(events)  # advance after processing
```

---

### Gotchas to not forget:

```
❌  More consumers than partitions — extra consumers sit idle (1 consumer max per partition)
❌  Using processing time for windows with out-of-order events — late data falls in wrong window
❌  Committing offsets before processing — event lost if consumer crashes
❌  Global ordering across partitions — Kafka only guarantees order WITHIN a partition
❌  Unbounded state store — expire/evict old keys or you OOM eventually
❌  Forgetting consumer lag monitoring — silent backlog growth is a production incident
✅  Use event_time + watermarks for window correctness with late data
✅  Dedup key at the sink = effective exactly-once without transaction overhead
✅  Consumer lag = producer offset − consumer committed offset (watch this metric!)
✅  DLQ is your safety net — never silently drop events in financial/audit pipelines
✅  Partition count can only increase — plan ahead (changing it reshuffles data)
```

<a id='12'></a>
## 12. Summary Map

---

```
                      🌊 STREAM PROCESSING
                             │
           ┌─────────────────┼──────────────────────┐
           │                 │                      │
    KAFKA ARCHITECTURE   DELIVERY SEMANTICS      WINDOWING
    (Pattern 1)          (Pattern 2)             (Pattern 3)
           │                 │                      │
    ┌──────┴──────┐   ┌──────┴──────┐      ┌────────┴────────┐
  Topics       Consumer  At-Most   Exactly  Tumbling  Sliding  Session
  Partitions   Groups    Once      Once     (fixed)  (overlap) (gap)
  Offsets      Rebalance At-Least  Transact
               Replay    Once      Idem.
           │
    ┌──────┴────────────────────────┐
    │                               │
  FRAMEWORKS (Pattern 4)     BACKPRESSURE (Pattern 5)
    │                               │
  ┌─┴──────────────┐       ┌────────┴──────────────┐
  Kafka Streams  Flink   Buffer   Drop   DLQ   Rate-Limit
  (embedded,    (event  (bounded  (load  (safe  (token
   KTable)       time,   queue)   shed)  drop)   bucket)
  Spark SS       wmarks)
  (micro-batch)

COMMON INTERVIEW FLOW:

  "Design a real-time analytics pipeline"

  Step 1: Kafka topic — partition by user_id for ordering
  Step 2: Consumer group — at-least-once + idempotent sink
  Step 3: Flink — tumbling windows (per minute), watermarks for late data
  Step 4: Serving layer — push aggregates to Redis / ClickHouse
  Step 5: Observability — consumer lag, drop rate, window lateness metrics
  Step 6: Backpressure — bounded buffer + DLQ; scale partitions if lag grows

KEY NUMBERS TO REMEMBER:
  Kafka throughput:   ~1 million msg/sec per broker
  Partition limit:    typically 100-4000 per cluster (depends on broker count)
  Replication factor: 3 (standard) — tolerates 2 broker failures
  Consumer lag SLA:   < 30 seconds for real-time; minutes for near-real-time
  EOS overhead:       ~20% throughput reduction vs at-least-once
```

---
*End of Stream Processing Master Guide — Sean Edition*